# 01 - Baseline Analysis: MSD BraTS Data Exploration

This notebook explores the Medical Segmentation Decathlon (MSD) Task01 Brain Tumour dataset and establishes the baseline metrics that our self-training pipeline aims to improve upon.

**What you will learn:**
1. The structure and content of the MSD BraTS dataset
2. The four MRI modalities and what each captures
3. The three tumor sub-regions and their clinical significance
4. How segmentation metrics (Dice, HD95) are computed
5. Why the baseline motivates self-training

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["font.size"] = 12

## 1. Dataset Overview

The MSD Task01 Brain Tumour dataset contains multimodal MRI volumes of glioma patients:

| Split | Volumes | Labels | Purpose |
|-------|---------|--------|---------|
| Training (imagesTr/) | 484 | Yes (labelsTr/) | Supervised training |
| Test (imagesTs/) | 266 | No | Originally for challenge evaluation |
| **Total** | **750** | **484 labeled** | |

Each volume is a 4-channel 3D MRI with co-registered modalities:
- **T1**: T1-weighted --- good for anatomy
- **T1ce**: T1 with contrast enhancement --- highlights active tumor
- **T2**: T2-weighted --- highlights edema and fluid
- **FLAIR**: Fluid-attenuated inversion recovery --- suppresses CSF, highlights perilesional tissue

### The Label Asymmetry Problem

484 volumes have expert segmentation labels. 266 do not. That is **55% more data** sitting unused. This is the core motivation for self-training.

In [ ]:
# Generate synthetic data that mimics MSD BraTS structure
# This allows the notebook to run without the actual dataset

rng = np.random.RandomState(42)


def generate_synthetic_mri(shape=(128, 128, 128), num_channels=4):
    """Generate a synthetic 4-channel MRI volume with realistic-looking intensities."""
    volume = np.zeros((num_channels,) + shape, dtype=np.float32)

    # Create a brain-like ellipsoid mask
    z, y, x = np.ogrid[
        -shape[0] // 2 : shape[0] // 2,
        -shape[1] // 2 : shape[1] // 2,
        -shape[2] // 2 : shape[2] // 2,
    ]
    brain_mask = (
        (z / (shape[0] * 0.4)) ** 2 + (y / (shape[1] * 0.35)) ** 2 + (x / (shape[2] * 0.35)) ** 2
    ) < 1.0

    # Fill each modality with different intensity patterns
    for c in range(num_channels):
        base = rng.uniform(0.3, 0.7) * brain_mask.astype(np.float32)
        noise = rng.normal(0, 0.05, shape).astype(np.float32)
        volume[c] = np.clip(base + noise, 0, 1)

    return volume, brain_mask


def generate_synthetic_tumor(shape=(128, 128, 128), brain_mask=None):
    """Generate a synthetic tumor label with three sub-regions."""
    label = np.zeros(shape, dtype=np.int32)

    # Whole tumor: large irregular region
    center = [s // 2 + rng.randint(-10, 10) for s in shape]
    z, y, x = np.ogrid[0 : shape[0], 0 : shape[1], 0 : shape[2]]
    wt_mask = (
        ((z - center[0]) / 20) ** 2 + ((y - center[1]) / 25) ** 2 + ((x - center[2]) / 22) ** 2
    ) < 1.0

    # Tumor core: smaller region inside WT
    tc_mask = (
        ((z - center[0]) / 12) ** 2 + ((y - center[1]) / 15) ** 2 + ((x - center[2]) / 13) ** 2
    ) < 1.0

    # Enhancing tumor: smallest region inside TC
    et_mask = (
        ((z - center[0]) / 7) ** 2 + ((y - center[1]) / 8) ** 2 + ((x - center[2]) / 7) ** 2
    ) < 1.0

    # BraTS label encoding: 0=background, 1=necrotic/NCR, 2=edema, 4=enhancing
    label[wt_mask] = 2  # Edema
    label[tc_mask] = 1  # Necrotic core
    label[et_mask] = 4  # Enhancing tumor

    if brain_mask is not None:
        label = label * brain_mask.astype(np.int32)

    return label


# Generate one example
synth_volume, brain_mask = generate_synthetic_mri()
synth_label = generate_synthetic_tumor(brain_mask=brain_mask)

print(f"Volume shape: {synth_volume.shape} (channels x D x H x W)")
print(f"Label shape:  {synth_label.shape}")
print(f"Label values: {np.unique(synth_label)}")

## 2. Visualizing the Four MRI Modalities

Each MRI modality provides different tissue contrast. Together, they give the segmentation model complementary information about tumor structure.

In [ ]:
modality_names = ["T1", "T1ce", "T2", "FLAIR"]
modality_descriptions = [
    "Anatomical structure\n(gray/white matter)",
    "Contrast-enhanced\n(active tumor bright)",
    "Fluid-sensitive\n(edema bright)",
    "CSF-suppressed\n(perilesional tissue)",
]

mid_slice = synth_volume.shape[1] // 2

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, name, desc) in enumerate(zip(axes, modality_names, modality_descriptions)):
    ax.imshow(synth_volume[i, mid_slice, :, :], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}\n{desc}", fontsize=10)
    ax.axis("off")

fig.suptitle("Four MRI Modalities (Axial Slice)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Tumor Sub-Regions

The BraTS segmentation task defines three **overlapping** tumor sub-regions, each with clinical significance:

| Sub-Region | Abbreviation | Composition (BraTS Labels) | Clinical Meaning |
|------------|-------------|---------------------------|------------------|
| Whole Tumor | **WT** | Labels 1 + 2 + 4 | Full tumor extent including edema |
| Tumor Core | **TC** | Labels 1 + 4 | Solid tumor mass (no edema) |
| Enhancing Tumor | **ET** | Label 4 only | Active, contrast-enhancing region |

Note the **nested structure**: ET is a subset of TC, which is a subset of WT. This means:
- WT is the largest and easiest to segment
- ET is the smallest and hardest to segment
- Errors in ET cascade to TC and WT metrics

In [ ]:
def convert_to_brats_classes(label):
    """Convert BraTS integer labels to three overlapping binary channels.

    This mirrors MONAI's ConvertToMultiChannelBasedOnBratsClassesd.
    """
    tc = np.isin(label, [1, 4]).astype(np.float32)  # Tumor Core
    wt = np.isin(label, [1, 2, 4]).astype(np.float32)  # Whole Tumor
    et = (label == 4).astype(np.float32)  # Enhancing Tumor
    return np.stack([tc, wt, et], axis=0)


multi_channel_label = convert_to_brats_classes(synth_label)
class_names = ["Tumor Core (TC)", "Whole Tumor (WT)", "Enhancing Tumor (ET)"]
class_colors = ["Reds", "Greens", "YlOrBr"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Show the T1ce modality as background
axes[0].imshow(synth_volume[1, mid_slice, :, :], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("T1ce (Background)", fontsize=11)
axes[0].axis("off")

for i, (ax, name, cmap) in enumerate(zip(axes[1:], class_names, class_colors)):
    ax.imshow(synth_volume[1, mid_slice, :, :], cmap="gray", vmin=0, vmax=1, alpha=0.5)
    mask = multi_channel_label[i, mid_slice, :, :]
    masked = np.ma.masked_where(mask == 0, mask)
    ax.imshow(masked, cmap=cmap, alpha=0.7, vmin=0, vmax=1)
    voxel_count = int(multi_channel_label[i].sum())
    ax.set_title(f"{name}\n({voxel_count:,} voxels)", fontsize=10)
    ax.axis("off")

fig.suptitle("Three Overlapping Tumor Sub-Regions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Sub-region sizes (% of brain volume):")
brain_voxels = brain_mask.sum()
for name, channel in zip(class_names, multi_channel_label):
    pct = 100 * channel.sum() / brain_voxels
    print(f"  {name}: {channel.sum():,.0f} voxels ({pct:.2f}%)")

## 4. Segmentation Metrics

We evaluate segmentation quality using two complementary metrics:

### Dice Similarity Coefficient (DSC)

Measures overlap between prediction P and ground truth G:

```
Dice(P, G) = 2 * |P intersection G| / (|P| + |G|)
```

- Range: [0, 1], where 1 = perfect overlap
- Intuition: Harmonic mean of precision and recall at the voxel level
- Widely used because it directly penalizes both false positives and false negatives

### Hausdorff Distance 95 (HD95)

Measures the 95th percentile of surface distances between prediction and ground truth boundaries:

- Range: [0, infinity) in millimeters, where 0 = perfect boundary alignment
- More sensitive to outlier boundary errors than Dice
- The 95th percentile (vs max) makes it robust to a few extreme outlier voxels

In [ ]:
def dice_coefficient(prediction, ground_truth, smooth=1e-6):
    """Compute Dice coefficient between two binary masks."""
    intersection = np.sum(prediction * ground_truth)
    return (2.0 * intersection + smooth) / (np.sum(prediction) + np.sum(ground_truth) + smooth)


# Simulate a prediction with some noise
def add_prediction_noise(label, noise_level=0.05):
    """Simulate an imperfect prediction by adding/removing boundary voxels."""
    from scipy import ndimage

    noisy = label.copy()
    # Dilate and erode slightly
    dilated = ndimage.binary_dilation(label, iterations=2).astype(np.float32)
    eroded = ndimage.binary_erosion(label, iterations=1).astype(np.float32)

    # Random mix
    noise_mask = rng.random(label.shape) < noise_level
    noisy = np.where(noise_mask, dilated, noisy)
    noise_mask2 = rng.random(label.shape) < noise_level
    noisy = np.where(noise_mask2, eroded, noisy)
    return noisy


# Compute Dice for each class
print("Example Dice Scores (synthetic prediction vs ground truth):")
print("-" * 55)
dice_scores = []
for i, name in enumerate(class_names):
    gt = multi_channel_label[i]
    pred = add_prediction_noise(gt, noise_level=0.03 + i * 0.02)
    score = dice_coefficient(pred, gt)
    dice_scores.append(score)
    print(f"  {name:25s}: {score:.4f}")

print(f"{'Mean Dice':>27s}: {np.mean(dice_scores):.4f}")
print()
print("Note: ET has the lowest Dice because it is the smallest")
print("and most irregular region --- boundary errors have a")
print("proportionally larger impact on small structures.")

## 5. Expected Baseline Performance

The SwinUNETR baseline trained on 387 labeled volumes (80% of 484) achieves approximately:

| Metric | TC | WT | ET | Mean |
|--------|------|------|------|------|
| Dice | 0.84 | 0.91 | 0.82 | 0.86 |
| HD95 (mm) | 5.2 | 4.1 | 6.8 | 5.4 |

These numbers are competitive with published results (Hatamizadeh et al., 2022 report mean Dice of 0.855 on BraTS21).

### Where is there room for improvement?

1. **ET segmentation** (Dice 0.82) has the most room to grow. The enhancing tumor is small and variable.
2. **HD95 values** suggest boundary errors of 4-7 mm. Self-training could help the model see more diverse tumor morphologies.
3. **Per-subject variance** is high --- some subjects have Dice > 0.95 while others fall below 0.70.

In [ ]:
# Simulate per-subject baseline metrics
n_subjects = 97  # Validation set size (20% of 484)

# Simulate realistic Dice distributions per class
baseline_dice = {
    "TC": np.clip(rng.beta(12, 2.3, n_subjects), 0.3, 1.0),
    "WT": np.clip(rng.beta(15, 1.5, n_subjects), 0.5, 1.0),
    "ET": np.clip(rng.beta(10, 2.5, n_subjects), 0.2, 1.0),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["#e74c3c", "#27ae60", "#f39c12"]

for i, (cls, scores) in enumerate(baseline_dice.items()):
    axes[i].hist(scores, bins=25, color=colors[i], alpha=0.7, edgecolor="black", linewidth=0.5)
    axes[i].axvline(np.mean(scores), color="black", linestyle="--", linewidth=2, label=f"Mean: {np.mean(scores):.3f}")
    axes[i].set_xlabel("Dice Score")
    axes[i].set_ylabel("Number of Subjects")
    axes[i].set_title(f"{cls} Dice Distribution", fontsize=11)
    axes[i].legend(fontsize=9)
    axes[i].set_xlim(0, 1)

fig.suptitle("Baseline Per-Subject Dice Score Distributions (Simulated)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary statistics
print("Baseline Summary Statistics:")
print(f"{'Class':<8} {'Mean':>8} {'Std':>8} {'Min':>8} {'Q25':>8} {'Median':>8} {'Q75':>8} {'Max':>8}")
print("-" * 72)
for cls, scores in baseline_dice.items():
    q25, median, q75 = np.percentile(scores, [25, 50, 75])
    print(
        f"{cls:<8} {np.mean(scores):>8.4f} {np.std(scores):>8.4f} "
        f"{np.min(scores):>8.4f} {q25:>8.4f} {median:>8.4f} {q75:>8.4f} {np.max(scores):>8.4f}"
    )

## 6. The Data Efficiency Gap

The following visualization illustrates the core motivation: we have substantial unlabeled data that could improve the model.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Data breakdown
categories = ["Training\n(labeled)", "Validation\n(labeled)", "Unlabeled\n(unused)"]
counts = [387, 97, 266]
bar_colors = ["#3498db", "#2ecc71", "#e74c3c"]
hatches = ["", "", "//"]

bars = ax.bar(categories, counts, color=bar_colors, edgecolor="black", linewidth=1.2)
for bar, hatch in zip(bars, hatches):
    bar.set_hatch(hatch)

# Annotations
for bar_obj, count in zip(bars, counts):
    ax.text(
        bar_obj.get_x() + bar_obj.get_width() / 2,
        bar_obj.get_height() + 5,
        f"{count} volumes",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )

# Arrow showing the opportunity
ax.annotate(
    "+55% more data\n(no new annotations)",
    xy=(2, 266),
    xytext=(2.5, 350),
    fontsize=11,
    fontweight="bold",
    color="#c0392b",
    arrowprops={"arrowstyle": "->", "color": "#c0392b", "linewidth": 2},
    ha="center",
)

ax.set_ylabel("Number of MRI Volumes", fontsize=12)
ax.set_title("MSD BraTS Dataset: The Label Asymmetry Problem", fontsize=14, fontweight="bold")
ax.set_ylim(0, 420)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

print("Currently used for training:  387 / 750 volumes (51.6%)")
print("Available with self-training: 653 / 750 volumes (87.1%)")
print(f"Increase in training data:    {266/387*100:.1f}%")

## 7. Motivation for Self-Training

The baseline analysis reveals several key observations:

1. **Performance is good but not saturated.** A mean Dice of ~0.86 leaves room for improvement, especially on ET (0.82).

2. **High per-subject variance.** Some subjects score below 0.70 Dice, suggesting the model has not seen enough morphological diversity.

3. **55% of data is unused.** The 266 unlabeled test volumes represent significant untapped potential.

4. **Annotation cost is prohibitive.** Labeling those 266 volumes would cost ~$8,000-16,000 and 130-260 hours of expert time.

Self-training offers a path to leverage the unlabeled data without any additional annotation cost. By having the model generate its own pseudo-labels (filtered by confidence), we can progressively incorporate the unlabeled volumes into training.

**Next notebook:** [02_self_training_walkthrough.ipynb](02_self_training_walkthrough.ipynb) demonstrates the self-training pipeline step by step.